In [12]:
import pdfplumber
import re
from unidecode import unidecode

In [13]:
file_path = "../resources/GA_61IW_615000240_2S_2024-25.pdf"

In [14]:
def clean_table(table):
    cleaned_table = []
    for row in table:
        cleaned_row = []
        for cell in row:
            if isinstance(cell, str):
                cell = re.sub(r'\(?\bRA\d+\)?,?', '', cell)
                cell = re.sub(r'\[.*?\]\s*', '', cell)
                cell = cell.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ').strip()
                cell = cell.replace('.', '')
                cell = cell.replace('%', '')
                cell = re.sub(r'[\*+]', ' ', cell)
                cell = unidecode(cell)
                cell = cell.lower()
                cell = re.sub(r'\s+', ' ', cell)
            cleaned_row.append(cell)
        cleaned_table.append(cleaned_row)
    return cleaned_table

In [62]:

def concat_two_tables(table1, table2):
    if not table1:
        return table2
    if not table2:
        return table1

    print("TABBLE1[0]: ")
    print(table1[0])
    print("LEN TABLE[1]: " + str(len(table1[0])))
    print("TABBLE2[0]: ")
    print(table2[0])
    print("LEN TABLE[2]: " + str(len(table2[0])))
    if len(table1[0]) != len(table2[0]):
        raise ValueError(
            "Las tablas no tienen el mismo número de columnas y no se pueden concatenar., las columnas de la tabla 1 "
            "son: ",
            len(table1[0]), " y las columnas de la tabla 2 son: ", len(table2[0]))
        

    combined_table = table1 + table2
    return combined_table

In [73]:
def get_necessary_tables(file_path):
    final_tables = []
    continuous_assessment_header = ['sem', 'descripcion', 'modalidad', 'tipo', 'duracion', 'peso en la nota',
                                     'nota minima', 'competencias evaluadas']
    final_assessment_header = ['sem', 'descripcion', 'modalidad', 'tipo', 'duracion', 'peso en la nota', 'nota minima',
                               'competencias evaluadas']
    extraordinary_assessment_header = ['descripcion', 'modalidad', 'tipo', 'duracion', 'peso en la nota', 'nota minima',
                                       'competencias evaluadas']
    
    #pdf_stream = BytesIO(file_bytes)

    with pdfplumber.open(file_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            tables = page.extract_tables()
            for table_number, table in enumerate(tables, start=1):
                cleaned_table = clean_table(table)
                if not cleaned_table:
                    continue
                if page_number == 1 and table_number == 1:
                    final_tables.append(cleaned_table)
                else:
                    if len(final_tables) == 1 and (cleaned_table[0] == continuous_assessment_header):
                        final_tables.append(cleaned_table)
                        print("CLEANED TABLE DE LA PRIMERA ITERACION ES : ")
                        print(cleaned_table)
                        print("EN ESTE PUNTO LA LONGITUD DE LA TABLA FINAL ES: " + str(len(final_tables)))
                        continue
                    if len(final_tables) == 2 and (cleaned_table[0] != final_assessment_header):
                        print("CLEANED TABLES ES: ") 
                        print(cleaned_table)
                        print("FINAL TABLE[1] ES:")
                        print(final_tables[1])
                        final_table = concat_two_tables(final_tables[1], cleaned_table)
                        final_tables[1] = final_table
                        continue
                    if len(final_tables) == 2 and (cleaned_table[0] == final_assessment_header):
                        final_tables.append(cleaned_table)
                        continue
                    if len(final_tables) == 3 and (cleaned_table[0] != extraordinary_assessment_header):
                        final_table = concat_two_tables(final_tables[2], cleaned_table)
                        final_tables[2] = final_table
                        print("RESULTADOSSSS")
                        print(final_tables[0])
                        print(final_tables[1])
                        print(final_tables[2])
                        continue
                    if len(final_tables) == 3 and (cleaned_table[0] == extraordinary_assessment_header):
                        final_tables.append(cleaned_table)
                        continue
                    if len(final_tables) == 4 and (cleaned_table[0] != extraordinary_assessment_header) and len(cleaned_table[0]) == 7:
                        final_table = concat_two_tables(final_tables[3], cleaned_table)
                        final_tables[3] = final_table
                        continue
                continue
        return final_tables

In [74]:
final_tables = get_necessary_tables(file_path)
print("TRUE RESULTSSS")
print(final_tables[0])
print(final_tables[1])
print(final_tables[2])
print(final_tables[3])


CLEANED TABLE DE LA PRIMERA ITERACION ES : 
[['sem', 'descripcion', 'modalidad', 'tipo', 'duracion', 'peso en la nota', 'nota minima', 'competencias evaluadas'], ['6', 'practica 1', 'tg: tecnica del tipo trabajo en grupo', 'no presencial', '00:00', '12', '/ 10', 'cb5 cc16'], ['7', 'examen parcial 1', 'ex: tecnica del tipo examen escrito', 'presencial', '01:00', '14', '/ 10', 'cc17 cb5 cc8 cc16'], ['15', 'examen parcial 2', 'ex: tecnica del tipo examen escrito', 'presencial', '01:00', '14', '/ 10', 'cc8 cc16 cc17 cb5'], ['17', 'practica 2', 'tg: tecnica del tipo trabajo en grupo', 'no presencial', '00:00', '18', '/ 10', 'cc17 ct8 cb5 cc8 ct11 cc16'], ['17', 'examen final', 'ex: tecnica del tipo examen escrito', 'presencial', '03:00', '42', '4 / 10', 'cc17 cb5 cc8 cc16']]
EN ESTE PUNTO LA LONGITUD DE LA TABLA FINAL ES: 2
TABBLE1[0]: 
['sem', 'descripcion', 'modalidad', 'tipo', 'duracion', 'peso en la nota', 'nota minima', 'competencias evaluadas']
LEN TABLE[1]: 8
TABBLE2[0]: 
['17', 'exa